In [ ]:
# Setup — all imports live here so the notebook executes top-to-bottom
# without reimporting mid-notebook. When running headlessly (CI, HPC),
# uncomment the ``matplotlib.use('Agg')`` line *before* the pyplot import
# so figures never need an interactive display.
import dataclasses
import logging
import os
import sys
from pathlib import Path

# import matplotlib
# matplotlib.use('Agg')  # enable for headless runs
import matplotlib.pyplot as plt
import mne
import numpy as np
import seaborn as sns

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:  # noqa: B007
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

from independent_vector_analysis import iva_g  # noqa: E402
from IPython.display import Image, display  # noqa: E402
from sklearn.decomposition import PCA  # noqa: E402

from scripts.notebook_helpers import (  # noqa: E402
    WAVELET_FREQ_MAX,
    WAVELET_FREQ_MIN,
    WAVELET_N_FREQS,
    load_paired_condition_wavelets,
    resolve_notebook_wavelet_cache_dir,
    resolve_wavelet_dir,
)
from src.analysis import assr_trials as at  # noqa: E402
from src.analysis import iva_quality  # noqa: E402
from src.analysis.condition_tracks import ZSCORE_MODES  # noqa: E402
from src.analysis.iva_condition_comparison import (  # noqa: E402
    apply_component_signs,
    stack_conditions_on_subject_axis,
)
from src.analysis.wavelet_ica import (  # noqa: E402
    align_iva_component_signs,
    iva_component_patterns,
    zscore_by_time,
)
from src.definitions.constants import AssrEpoch, ProjectPaths  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    REAL_CONDITIONS,
    ConditionVariants,
    CoordinateSystems,
    ExclusionCategories,
    ExperimentNames,
    MusicTypeVariants,
)
from src.io.loading import assr_electrode_mask  # noqa: E402
from src.visualization.iva_condition_plots import (  # noqa: E402
    plot_condition_mean_tf_maps,
    plot_condition_mean_topomaps,
    plot_participant_condition_tf_maps,
    plot_participant_condition_topomaps,
)
from src.visualization.iva_quality_plots import topo_info_subset  # noqa: E402

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
mne.set_log_level("ERROR")
%matplotlib inline
print("Setup complete.")

# IVA Decomposition of Wavelet Power — Channel Components, Conditions Concatenated in Time

The **second** way to join the conditions, and the companion of
[`wavelet_iva_channel_joined.ipynb`](wavelet_iva_channel_joined.ipynb). Both run the
identical channel-as-independent IVA-G pipeline on the identical participants; they
differ only in **which axis the two conditions are pooled along**, and that one choice
changes what the comparison can say.

Here each participant is **one** subject whose recording is their Placebo track followed
by their Psilocybin track
([`ConditionVariants.JOINED_TRACKS`](../../src/definitions/fields.py)):

```
subject k  ->  [ ---- Placebo track ---- | ---- Psilocybin track ---- ]
               0                     T_pl                  T_pl + T_ps
```

```
datasets = participants        ← P entries, not 2P
samples  = time × frequency    ← time spans BOTH conditions
mixing   = channel             ← one matrix per participant, both conditions
```

## What this buys, and what it costs

**The decomposition is shared by construction.** The subject axis holds one entry per
participant and the mixing is estimated over the whole concatenated recording, so IVA
returns a *single* set of components covering both conditions. The contrast is then read
off by slicing the sources back into the two time segments — the components are the same
in both conditions **by construction**, so there is no component-matching step and no
risk of comparing a Placebo component against a different Psilocybin one.

**The cost is the pattern side.** One mixing matrix per participant means **one channel
topography per (participant, component), shared by both conditions.** There is
therefore *no per-condition topography to compare*: the contrast lives entirely in the
sources. Step 7 draws the topographies once, labelled as shared, rather than drawing two
identical rows and a zero difference. If a condition difference **in the scalp pattern**
is what you are after, that is exactly what the subject-axis variant is for — there each
recording gets its own mixing matrix, and
[`wavelet_iva_channel_joined.ipynb`](wavelet_iva_channel_joined.ipynb) compares them.

| | this notebook (`JOINED_TRACKS`) | [subject-axis](wavelet_iva_channel_joined.ipynb) (`JOINED`) |
|---|---|---|
| Pools along | **time** | subject axis |
| Subject axis | `P` (each participant once) | `2P` (each participant twice) |
| Mixing matrices | one per participant | one per **recording** |
| Components across conditions | identical by construction | free, aligned only via the shared source |
| TF-map contrast | ✔ split the time axis | ✔ split the subject axis |
| Topography contrast | ✘ shared by construction | ✔ |
| Needs a shared time base | no — tracks may differ in length | yes |

## Z-scoring happens *before* the concatenation

Each condition is standardised **on its own** and the standardised tracks are then laid
end to end (`ZSCORE_MODE = "per_condition"`, the default of
`concatenate_condition_tracks`). So each track enters the decomposition on exactly the
same footing as it would in a single-condition run.

Note the consequence, which is the same trade-off the subject-axis variant makes: every
segment then has zero mean and unit variance per `(participant, channel, frequency)` by
construction, so an **overall power difference between the conditions is normalised
away**. What the comparison tests is the temporal and spectral *structure* of each
condition, not its amplitude. Set `ZSCORE_MODE = "joint"` to keep the amplitude contrast
instead — see `ZSCORE_MODES`.

A neat consequence of `"per_condition"`: the pipeline's usual first step,
`zscore_by_time` on the concatenated array, becomes an **exact identity**. Two unit-
variance zero-mean segments concatenate to a series with mean 0 and variance
`(n₁·1 + n₂·1)/(n₁+n₂) = 1`. Step 1 applies it anyway — so this notebook's pipeline is
byte-for-byte the one the other variants run — and checks numerically that it changed
nothing.

## Reshape

```
Input:   (n_participants, n_channels, n_freqs, n_times_total)
Per-participant reshape:
         (n_channels,  n_freqs × n_times_total)
           ── mixing ──   ────── samples ──────
Per-participant PCA over channels → N_PCA
Stack:   (N_PCA, n_freqs × n_times_total, n_participants)  — IVA layout (N, T, K)
```

## Variables produced

| Variable | Shape | Description |
|----------|-------|-------------|
| `paired` | — | `PairedConditionTracks` — the concatenated dataset plus its segment bookkeeping |
| `bb_data` | `(P, C, F, T_total)` | Concatenated wavelet power, per-condition z-scored |
| `X_pca` | `(N_PCA, F·T_total, P)` | PCA-reduced IVA input layout |
| `W` | `(N_PCA, N_PCA, P)` | Per-participant IVA demixing matrices (sign-aligned) |
| `iva_sources` | `(P, N_PCA, F, T_total)` | Spectro-temporal sources, spanning both conditions |
| `iva_components` | `(P, N_PCA, C)` | Per-participant channel topographies — **shared by both conditions** |
| `sources_by_condition` | dict | `(P, N_PCA, F, T_c)` per condition, after the split |
| `sources_stacked` | `(2P, N_PCA, F, T_min)` | The split restacked on the subject axis, for the figures |

## Configuration

In [ ]:
# ── Experiment configuration ───────────────────────────────────
# Choose the experiment: ExperimentNames.PSILO_MUSIC or ExperimentNames.ASSR
EXPERIMENT_NAME = ExperimentNames.ASSR
# The concatenated dataset IS the condition here — the two real conditions below are
# laid end to end along time, one subject per participant.
CONDITION = ConditionVariants.JOINED_TRACKS
# Time-axis segment order: the Placebo track first, then Psilocybin.
CONDITIONS_TO_POOL = list(REAL_CONDITIONS)
if EXPERIMENT_NAME == ExperimentNames.ASSR:
    # ASSR has no music dimension; uses a single placeholder "music type".
    MUSIC_TYPE = MusicTypeVariants.ASSR
else:
    MUSIC_TYPE = MusicTypeVariants.CLASSICAL
EXCLUSION_CATEGORIES = [ExclusionCategories.BAD_MUSIC, ExclusionCategories.ARTIFACTS]

# ── Wavelet settings ─────────────────────────────────────────
REPRESENTATION = "power"
# Wavelet frequency grid: canonical 1 Hz-spaced grid from src.definitions.frequency
# (WAVELET_FREQ_MIN/MAX/N_FREQS), re-exported via scripts.notebook_helpers. It must
# match the grid the cache was written with, or the cache is missed and recomputed.
FREQS = np.linspace(WAVELET_FREQ_MIN, WAVELET_FREQ_MAX, WAVELET_N_FREQS)

# ── Standardisation before the concatenation ─────────────────
# "per_condition" (default): standardise each track on its own, then concatenate — each
# enters the decomposition exactly as in a single-condition run, and an overall power
# difference between conditions is normalised away. "joint" concatenates first and
# z-scores over the whole recording, keeping that difference as a mean offset between
# the segments. "none" assumes the caller already standardised.
ZSCORE_MODE = "per_condition"
assert ZSCORE_MODE in ZSCORE_MODES, f"ZSCORE_MODE must be one of {ZSCORE_MODES}"

# ── Reuse / compute ──────────────────────────────────────────
# True is the intended setting: this notebook is built around reusing the two
# per-condition caches. False would recompute both from RAW_AFTER_ICA.
REUSE_WAVELETS = True

# ── Cohort, channel and time subset ────────────────────────────
# N_PAIRS_SUBSET counts participants, which here is the same as subjects: each
# participant is ONE subject carrying both tracks.
N_PAIRS_SUBSET: int | None = 5
# NOTE: the sample axis is F × T_total, and T_total spans BOTH conditions — so it is
# twice the single-condition length on top of the ``n_freqs`` factor. N_TIMES_SUBSET is
# applied per condition (per segment), so the concatenated axis is 2 × it.
N_CHANNELS_SUBSET: int | None = 32  # first N channels (from 195)
N_TIMES_SUBSET: int | None = 3000  # first N samples OF EACH CONDITION

# ── IVA settings ──────────────────────────────────────────────
# N_COMPONENTS_PCA reduces the *channel* axis here, so it must be <= n_channels
# (<= N_CHANNELS_SUBSET when set). 10 is the standard ASSR setting for this variant
# and the default of scripts/run_wavelet_iva_channel.py.
N_COMPONENTS_PCA = 10  # per-participant PCA dim over channels (= N in IVA's (N, T, K))
IVA_OPT_APPROACH = "newton"  # 'gradient', 'newton', or 'quasi'
IVA_MAX_ITER = 64
IVA_W_DIFF_STOP = 1e-6
IVA_VERBOSE = True
IVA_RANDOM_STATE = 42  # seeds per-participant PCA + W_init

# ── Comparison figures ────────────────────────────────────────
SAVE_PLOTS = True
# Which components the comparison figures cover. None = every component.
COMPONENTS_TO_PLOT: list[int] | None = None
# Reference lines on the TF panels. The ASSR is continuous 40 Hz stimulation, so the
# stimulation frequency is the row worth locating.
TF_FREQ_MARKS: list[float] = [40.0]
MARK_STIMULUS_ONSETS_ON_TF = True

# ── Stimulus-locked epoch (Step 9) ────────────────────────────
# The paradigm window from src.definitions.constants.AssrEpoch, so this notebook, the
# subject-axis variant, the 05 quality notebook and the headless --quality path all cut
# the SAME epoch. Applied to each condition's own split track, with its own onsets.
EPOCH_PRE_S = AssrEpoch.PRE_ONSET_S  # short interval before each stimulus
EPOCH_POST_S = AssrEpoch.POST_ONSET_S  # stimulus + post-stimulus
MIN_ONSETS_FOR_EPOCH_AVERAGE = 5

# ── Wavelet cache directories ─────────────────────────────────
# WAVELET_DIR is the source of truth (one big compressed file per condition);
# WAVELET_SUBSET_CACHE_DIR holds per-extent copies under the stage-03 notebook, shared
# with every other wavelet workflow. See the 03 README for the full comparison.
WAVELET_DIR: Path = resolve_wavelet_dir(None, EXPERIMENT_NAME) / "broadband"
WAVELET_SUBSET_CACHE_DIR: Path = (
    resolve_notebook_wavelet_cache_dir(EXPERIMENT_NAME) / "broadband"
)
REUSE_WAVELET_SUBSET_CACHE = True

# ── Plots directory ───────────────────────────────────────────
# Same stage as the subject-axis variant, but its own analysis_type subdirectory so the
# two sets of figures never collide.
PLOTS_DIR: Path = (
    ProjectPaths.NOTEBOOKS_DIR
    / "06-iva-condition-comparison"
    / "plots"
    / EXPERIMENT_NAME.value
    / "broadband"
    / "iva_channel_joined_tracks"
    / f"pca_{N_COMPONENTS_PCA}"
)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Experiment / group  : {EXPERIMENT_NAME.value} — {CONDITION.value}")
print(f"Segment order       : {[c.value for c in CONDITIONS_TO_POOL]} (time axis)")
print(f"Standardisation     : {ZSCORE_MODE} (before the concatenation)")
print(f"Wavelet source cache: {WAVELET_DIR}")
print(
    f"Wavelet subset cache: {WAVELET_SUBSET_CACHE_DIR}  "
    f"(reuse: {REUSE_WAVELET_SUBSET_CACHE})"
)
print(f"Frequencies         : {FREQS[0]:.1f}–{FREQS[-1]:.1f} Hz ({len(FREQS)} steps)")
print(f"Per-participant PCA : {N_COMPONENTS_PCA}  (over channels)")
print(f"IVA optimisation    : {IVA_OPT_APPROACH}  (max_iter={IVA_MAX_ITER})")
print(f"Plots directory     : {PLOTS_DIR}  (saving: {SAVE_PLOTS})")
print(
    f"Stimulus epoch      : [-{EPOCH_PRE_S}, {EPOCH_POST_S}] s around each onset "
    f"({AssrEpoch.STIMULUS_DURATION_S} s stimulus + "
    f"{AssrEpoch.POST_STIMULUS_S} s post-stimulus)"
)

## Data Loading — tracks concatenated per participant

One call does the whole join: load each condition's wavelet power from its existing
cache, keep the participants present in both, standardise each track, and lay them end
to end along time.

**Nothing upstream changes.** Each condition keeps its own alignment, which is only ever
used *within* that condition, so the two tracks need not share a time base and need not
even have the same length — no re-alignment, no new crop stage, no new cache. (This is
where the two variants genuinely differ in requirements: the subject-axis join *does*
need one shared time base, because it stacks recordings into one subject axis.)

**Memory note.** A cached wavelet tensor is decompressed in full before it is trimmed to
the requested channel/time extent, and the two conditions are loaded and trimmed **one
at a time**, so the peak is one condition's cache — the same peak as a single-condition
notebook. The trimmed result is then written to `WAVELET_SUBSET_CACHE_DIR`, so later runs
at the same extent skip that read entirely. Those cache entries are the same ones the
subject-axis notebook and the 03/04/05 workflows read.

In [ ]:
paired, condition_analyzers = load_paired_condition_wavelets(
    MUSIC_TYPE,
    EXCLUSION_CATEGORIES,
    FREQS,
    wavelet_dir=WAVELET_DIR,
    experiment_name=EXPERIMENT_NAME,
    representation=REPRESENTATION,
    conditions=CONDITIONS_TO_POOL,
    zscore_mode=ZSCORE_MODE,
    n_channels=N_CHANNELS_SUBSET,
    n_times=N_TIMES_SUBSET,
    reuse_wavelets=REUSE_WAVELETS,
    subset_cache_dir=WAVELET_SUBSET_CACHE_DIR,
    reuse_subset_cache=REUSE_WAVELET_SUBSET_CACHE,
)

print(f"Concatenated dataset : {paired.data.label}")
print(f"Participants         : {paired.n_pairs}  (= subjects; each carries both tracks)")
print(f"Shape                : {paired.data.data.shape}")
print(f"Segment lengths      : {paired.segment_lengths}  -> total {paired.total_length}")
print(f"Segment boundaries   : {paired.boundaries}")

# Restrict the cohort by participant. Here that is also the subject axis, so a leading
# slice would be correct — but going through the participant labels keeps this cell
# identical in meaning to the subject-axis notebook, where it would not be.
if N_PAIRS_SUBSET is not None and N_PAIRS_SUBSET < paired.n_pairs:
    keep = sorted(paired.participants)[:N_PAIRS_SUBSET]
    rows = [paired.participants.index(p) for p in keep]
    paired = dataclasses.replace(
        paired,
        data=dataclasses.replace(paired.data, data=paired.data.data[rows]),
        participants=tuple(keep),
    )
    print(f"\nUsing first {len(keep)} participant(s): {keep}")
    print(f"Shape                : {paired.data.data.shape}")

print(f"\nParticipants         : {list(paired.participants)}")

## Dataset Selection

Unpacks the concatenated dataset into the same names the other variants use (`bb_data`,
`sfreq`, `n_subjects`, …) and records the segment bookkeeping the split in Step 6 needs.

Note `n_subjects == paired.n_pairs`: one subject per participant. The per-condition
stimulus onsets are kept **local to their own segment**, which is what
`condition_track` returns, so they line up with the split tracks without shifting.

In [ ]:
LABEL = paired.data.label

bb_ad = paired.data
bb_data = bb_ad.data  # (n_participants, n_channels, n_freqs, n_times_total)
sfreq = bb_ad.sfreq

n_subjects, n_channels, n_freqs, n_times_total = bb_data.shape
time_total = np.arange(n_times_total) / sfreq

participants = list(paired.participants)
CONDITION_ROWS = [c.value for c in paired.conditions]
# Per-condition onsets, local to that condition's own segment.
onsets_by_condition = {
    c: paired.condition_onsets(c, absolute=False) for c in paired.conditions
}

print(f"Dataset      : {LABEL}")
print(f"Shape        : {bb_data.shape}  (participants × channels × freqs × times)")
print(f"Duration     : {n_times_total / sfreq:.1f} s total  @  {sfreq} Hz")
print(f"Freq range   : {FREQS[0]:.1f}–{FREQS[-1]:.1f} Hz ({n_freqs} steps)")
print(f"Participants : {n_subjects}  (one subject each)")
for condition in paired.conditions:
    segment = paired.segment(condition)
    n_onsets = onsets_by_condition[condition]
    print(
        f"  {condition.value:<12}: samples [{segment.start}, {segment.stop}) "
        f"= {(segment.stop - segment.start) / sfreq:.1f} s, "
        f"{0 if n_onsets is None else len(n_onsets)} onset(s)"
    )

# MNE info for the topomaps, restricted to the IVA channel subset. Both conditions were
# preprocessed onto the same montage, so either analyser will do.
iva_info = topo_info_subset(
    condition_analyzers[paired.conditions[0]].info, n_channels
)
print(f"Topomap channels : {len(iva_info['ch_names'])} "
      f"(first={iva_info['ch_names'][0]}, last={iva_info['ch_names'][-1]})")

---
## Step 1 — Z-score and Per-Participant Reshape

Applied for symmetry with the other variants, so this notebook runs the identical
pipeline. With `ZSCORE_MODE = "per_condition"` it is an **exact identity**: each track
was already standardised before the concatenation, and two unit-variance zero-mean
segments concatenate to a series with mean 0 and variance 1. The cell checks that
numerically rather than asserting it in prose — under `"joint"` or `"none"` it is a real
normalisation and the printed deviation will be non-zero.

**Reshape** is done independently per participant: each `(C, F, T_total)` slice becomes a
`(C, F·T_total)` matrix — **channels** form the mixing axis and the joint
**(frequency, time)** axis forms the samples, with time now spanning both conditions. The
flatten keeps frequency slow and time fast (`index = f·T + t`), so the sample axis
reshapes cleanly back to `(F, T_total)` after IVA — and only then is it safe to split.

In [ ]:
bb_z = zscore_by_time(bb_data)  # (P, C, F, T_total)

# With "per_condition" this changed nothing; say so with a number, not a claim.
_max_dev = float(np.abs(bb_z - bb_data).max())
_scale = float(np.abs(bb_data).max())
print(f"z-score max |change| : {_max_dev:.3e}  (data |max| {_scale:.3g})")
if ZSCORE_MODE == "per_condition":
    print(
        "  -> expected ~0: each track was standardised before the concatenation, so "
        "the concatenated series already has mean 0 and unit variance."
    )
    assert _max_dev < 1e-8 * max(_scale, 1.0), (
        "zscore_by_time was expected to be an identity under 'per_condition' but "
        f"changed the data by {_max_dev:.3e}."
    )

# Per-participant reshape: (P, C, F, T) → (P, C, F*T), frequency-slow / time-fast.
n_samples_ft = n_freqs * n_times_total
X_subjects = bb_z.reshape(n_subjects, n_channels, n_samples_ft)

print(f"\nPer-participant reshape: {X_subjects.shape}  (participants, C, F*T_total)")
print(f"  Mixing dim (channels) : {n_channels}")
print(f"  Samples per subject   : {n_samples_ft}  (F={n_freqs} * T={n_times_total})")
print(f"  Datasets (participants): {n_subjects}  — both conditions inside each")

---
## Step 2 — Per-Participant PCA Over Channels

`iva_g` assumes a **square** mixing matrix per dataset. The mixing axis is **channels**,
so each participant's `(C, F·T_total)` matrix is reduced with its **own PCA over
channels** to `N_PCA` channel-mixtures before the K participant matrices are stacked.

This is where the two variants part company. Here a participant contributes **one**
matrix covering both of their tracks, so it gets **one** PCA and later **one** mixing
matrix — which is exactly why the recovered topography is shared between the conditions.
In the subject-axis variant the same participant contributes two separate datasets and
gets two of each.

In [ ]:
if N_COMPONENTS_PCA > n_channels:
    raise ValueError(
        f"N_COMPONENTS_PCA ({N_COMPONENTS_PCA}) must be <= n_channels "
        f"({n_channels}); PCA reduces the channel axis in this notebook."
    )

pcas: list[PCA] = []
pca_scores_per_subject = np.zeros((n_subjects, N_COMPONENTS_PCA, n_samples_ft))
pca_evr = np.zeros((n_subjects, N_COMPONENTS_PCA))

for k in range(n_subjects):
    subj_matrix = X_subjects[k].T  # (F*T, C) — samples x channels for sklearn
    pca = PCA(n_components=N_COMPONENTS_PCA, random_state=IVA_RANDOM_STATE)
    scores = pca.fit_transform(subj_matrix)  # (F*T, N_PCA)
    pcas.append(pca)
    pca_scores_per_subject[k] = scores.T
    pca_evr[k] = pca.explained_variance_ratio_
    print(
        f"  {participants[k]:<6}: explained variance = "
        f"{pca_evr[k].sum() * 100:5.1f}% ({N_COMPONENTS_PCA} channel comps)"
    )

X_pca = np.ascontiguousarray(pca_scores_per_subject.transpose(1, 2, 0))
print(f"\nIVA input shape        : {X_pca.shape}  (N_PCA, F*T_total, K=participants)")

fig, ax = plt.subplots(figsize=(8, 4))
for k in range(n_subjects):
    ax.plot(
        np.arange(1, N_COMPONENTS_PCA + 1),
        np.cumsum(pca_evr[k]),
        marker="o",
        markersize=3,
        label=participants[k],
    )
ax.set_xlabel("Number of PCA components (channels)")
ax.set_ylabel("Cumulative variance explained")
ax.set_title(f"Per-Participant Channel PCA — {LABEL}")
ax.axhline(0.9, ls="--", lw=0.6, color="gray")
ax.legend(fontsize=8, ncol=2, title="Participant")
fig.tight_layout()
plt.show()
plt.close("all")

---
## Step 3 — Run IVA-G

`iva_g` returns a demixing matrix `W` of shape `(N, N, K)` with `K = P` participants.
IVA's permutation ambiguity is shared across datasets, so the kth source in participant
0 is the kth source in participants `1..P-1` — no post-hoc matching.

`Sigma_N[:, :, k]` is the kth SCV's covariance across participants, computed over the
**whole concatenated** `F·T_total` axis, i.e. over both conditions at once. It says how
strongly a component couples across people; it says nothing about the conditions, which
is what Step 6 onward is for.

**Topographies are the forward (mixing) patterns**, i.e.
`pinv(W_k @ pcas[k].components_)` — *not* the unmixing rows. `iva_g` folds its internal
whitening into the returned `W_k`, so the unmixing rows carry an extra `Σ⁻¹` weighting;
the filter and the pattern of the same component can be nearly uncorrelated.

In [ ]:
rng = np.random.default_rng(IVA_RANDOM_STATE)
W_init = rng.standard_normal((N_COMPONENTS_PCA, N_COMPONENTS_PCA, n_subjects))

W, cost, Sigma_N, isi = iva_g(
    X_pca,
    opt_approach=IVA_OPT_APPROACH,
    whiten=True,
    verbose=IVA_VERBOSE,
    W_init=W_init,
    max_iter=IVA_MAX_ITER,
    W_diff_stop=IVA_W_DIFF_STOP,
)

print(f"\nW shape           : {W.shape}  (N_PCA, N_PCA, K=participants)")
print(f"Sigma_N shape     : {Sigma_N.shape}  (K, K, N_PCA)")
print(f"Iterations        : {len(cost)}")
print(f"Final cost        : {cost[-1]:.6f}")
if len(cost) >= IVA_MAX_ITER:
    print(f"  WARNING: hit max_iter={IVA_MAX_ITER}; W may not have converged "
          f"(W_diff_stop={IVA_W_DIFF_STOP}).")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(cost, marker="o", markersize=3, color="steelblue")
ax.set_xlabel("Iteration")
ax.set_ylabel("IVA cost")
ax.set_title(f"IVA-G Convergence — {LABEL}")
fig.tight_layout()
plt.show()
plt.close("all")

In [ ]:
# Resolve IVA's per-participant sign ambiguity from Sigma_N: each component is
# recovered only up to a per-dataset sign, so participant i's copy may be the negation
# of participant j's, which makes shared activity look anti-correlated. Step 5 then
# re-decides the orientation from the maps themselves.
#
# Note what a "dataset" is here: one participant, both tracks. So a flip moves that
# participant's WHOLE concatenated recording — it can never flip their Placebo segment
# relative to their Psilocybin one. That is what keeps the two conditions comparable
# after the split.
sigma_corr, W, sign_flips = align_iva_component_signs(Sigma_N, W)
print(
    f"Sigma_N sign alignment: flipped {int((sign_flips < 0).sum())} "
    f"(component, participant) pairs across {N_COMPONENTS_PCA} components."
)
print(f"sigma_corr shape  : {sigma_corr.shape}  (N_PCA, K, K)")

---
## Step 4 — Recover Spectro-Temporal Sources and Channel Patterns

Apply each participant's demixing matrix to obtain the IVA scores on the aligned
`F·T_total` axis, reshape them back to `(F, T_total)`, and combine `W_k` with the PCA
loadings to express each component's channel topography `(C,)`.

The sources still span **both** conditions at this point — the split comes in Step 6,
after the sign is settled.

In [ ]:
iva_scores_pca = np.zeros((n_subjects, N_COMPONENTS_PCA, n_samples_ft))
iva_components = np.zeros((n_subjects, N_COMPONENTS_PCA, n_channels))

for k in range(n_subjects):
    W_k = W[:, :, k]
    iva_scores_pca[k] = W_k @ X_pca[:, :, k]
    # Forward (mixing) patterns in channel space — NOT the unmixing rows.
    iva_components[k] = iva_component_patterns(W_k, pcas[k].components_)

# Reshape the aligned F*T axis back to (F, T_total). Flatten was frequency-slow,
# time-fast, so this is exact — and it has to be, or the split below would cut the
# wrong samples.
iva_sources = iva_scores_pca.reshape(
    n_subjects, N_COMPONENTS_PCA, n_freqs, n_times_total
)

print(f"IVA sources (F, T_total) : {iva_sources.shape}  (P, N_PCA, F, T_total)")
print(f"IVA channel patterns     : {iva_components.shape}  (P, N_PCA, C)")
print("  ^ one topography per (participant, component) — SHARED by both conditions")

---
## Step 5 — Components arrive with an arbitrary per-participant sign

`iva_g` fixes each component only up to a per-dataset sign, so participant *i*'s copy of
component *k* may be the negation of participant *j*'s. Averaging unresolved maps drives the
group mean toward zero, and on a pooled dataset it is worse than a lost mean: the flips
fall arbitrarily across the two condition blocks, so an unresolved sign manufactures a
condition difference out of nothing.

Only bookkeeping happens here — the sign itself is resolved in **Step 5b**, against the
ASSR electrode topography. An earlier version of this notebook resolved it here from PC1
of the TF maps instead; that made a component's sign *consistent* in the sense PC1
defines, which is not the same as "positive means more power over the ASSR area" — and
the latter is what every contrast and figure downstream actually needs. Anchoring
directly to the topography delivers it in one step, so the PC1 pass has been dropped
rather than composed with it.


In [ ]:
# Which components the comparison figures draw. The SIGN is not decided here — Step 5b
# anchors it to the ASSR electrodes, which is the only alignment this notebook applies.
COMP_INDICES = (
    list(range(N_COMPONENTS_PCA)) if COMPONENTS_TO_PLOT is None else COMPONENTS_TO_PLOT
)
print(f"Components in the comparison figures: {[k + 1 for k in COMP_INDICES]}")
print(
    f"Signs still unresolved at this point: {n_subjects} participants x "
    f"{N_COMPONENTS_PCA} components, each +-1 as iva_g happened to return it."
)


---
## Step 5b — Anchor every component's sign to the ASSR electrodes

The alignment above makes a component's sign **consistent** across participants, in the sense
PC1 of the TF maps defines. That is not the same as "positive means more power over the
ASSR area", and it is the second property every downstream figure and test depends on:
without it a positive value means *more* power there for some participants and *less* for
others, and a group mean partly cancels.

This step fixes that, per **(participant, component)** — each component carries its own
independent sign, so one flip shared across a participant's components would be wrong.

The anchor is `corr(topography, 0/1 ASSR mask)` taken across the whole channel axis.
Since `cov(pattern, mask) = f(1-f)(mean_inside - mean_outside)` for a binary mask, that
asks whether the pattern is more positive over the ASSR area **than over the rest of the
head** — so a pattern riding on a global offset cannot flip it, which an anchor reading
only the mean *inside* the mask cannot guarantee. It reads the topography alone and
never the response, so it stays symmetric in the conditions and cannot manufacture a
contrast.

Every pair is flipped on the sign of that correlation however small it is: declining to
flip a weak one does not avoid a choice, it just keeps the run's arbitrary sign instead.
The weak ones earn a printed count, because a wrongly flipped participant *cancels* signal in
a group mean rather than merely widening it.

This matches what `scripts/run_iva_condition_tracks.py` writes into the store, so the
figures below are oriented the same way as the arrays every later analysis reads.


In [ ]:
# The 0/1 electrode selection, on this run's own channel axis.
ANCHOR_COORDINATE_SYSTEM = CoordinateSystems.HYDROGEL_257_NO_FIDUCIALS
ANCHOR_MASK_STRICT = True

assr_anchor_mask = assr_electrode_mask(
    list(iva_info["ch_names"]), ANCHOR_COORDINATE_SYSTEM, strict=ANCHOR_MASK_STRICT
)

mask_flip, mask_strength = at.polarity_flip(
    iva_components, assr_anchor_mask
)

# Applied to every quantity carrying the per-(participant, component) sign, exactly
# as the PC1 pass was — topography and map must agree about which way is up.
iva_sources = apply_component_signs(iva_sources, mask_flip)
iva_components = apply_component_signs(iva_components, mask_flip)
iva_scores_pca = iva_sources.reshape(n_subjects, N_COMPONENTS_PCA, n_samples_ft)

n_weak, n_total = at.polarity_weak_count(mask_strength)
print(
    f"ASSR-mask sign anchor over {int(assr_anchor_mask.sum())} electrode(s): "
    f"{int((mask_flip < 0).sum())}/{mask_flip.size} (participant, component) "
    "pair(s) flipped."
)
print("\n  IC   flipped   median |corr|   weak")
for k in range(N_COMPONENTS_PCA):
    weak_k = int((mask_strength[:, k] < at.POLARITY_CORR_FLOOR).sum())
    print(
        f"  {k + 1:>3}   {int((mask_flip[:, k] < 0).sum()):>7}   "
        f"{np.median(mask_strength[:, k]):>13.3f}   "
        f"{f'{weak_k}/{mask_flip.shape[0]}':>4}"
    )
print(
    f"\n  {n_weak}/{n_total} pair(s) decided on |corr| < "
    f"{at.POLARITY_CORR_FLOOR}. Still flipped — the alternative is the run's\n"
    "  own arbitrary sign, not a safer one — but a wrongly flipped participant\n"
    "  CANCELS signal in a group mean rather than merely widening it."
)

# The note every figure below prints, now naming the anchor actually in force.
ALIGNMENT_NOTE = "corr(topography, ASSR electrode mask), per (participant, component)"


---
## Step 6 — Split the Time Axis Back Into the Two Conditions

This is the payoff of the whole construction, and it is one slice.
`paired.condition_track(array, condition)` cuts any array whose **last** axis is the
concatenated time axis, so it works unchanged on the sources `(P, K, F, T_total)`. It
validates that last axis, so passing something that did not come from this concatenation
raises rather than silently mis-cutting — which is why the channel patterns
`(P, K, C)` cannot be passed through it by accident.

The comparison figures index a single `(S, K, ...)` array by per-recording bookkeeping —
the layout the subject-axis variant produces natively. `stack_conditions_on_subject_axis`
restacks the split into exactly that, so **one set of figures serves both variants**.
The conditions keep their own time bases here and may differ in length, so the stack is
trimmed to the shorter one; each condition's full-length track stays available in
`sources_by_condition`.

In [ ]:
sources_by_condition = {
    c.value: paired.condition_track(iva_sources, c) for c in paired.conditions
}
for name, arr in sources_by_condition.items():
    print(f"{name:<12}: {arr.shape}  (P, N_PCA, F, T_condition)")

# Restack onto the subject axis the figures index by: Placebo block then Psilocybin.
sources_stacked, subject_participants, subject_conditions = (
    stack_conditions_on_subject_axis(
        sources_by_condition, CONDITION_ROWS, participants
    )
)
n_times_split = sources_stacked.shape[-1]
time_split = np.arange(n_times_split) / sfreq

print(f"\nStacked for the figures : {sources_stacked.shape}  (2P, N_PCA, F, T_min)")
print(f"Subject axis            : {list(zip(subject_participants, subject_conditions))}")
_lengths = {n: a.shape[-1] for n, a in sources_by_condition.items()}
if len(set(_lengths.values())) > 1:
    print(
        f"  NOTE: segments differ in length {_lengths}; the stack was trimmed to "
        f"{n_times_split} samples. Read sources_by_condition for the full tracks."
    )

# Onsets on each condition's own (trimmed) split axis, for the faint TF markers.
_ref_onsets = onsets_by_condition[paired.conditions[0]]
stimulus_onset_times = (
    np.array([])
    if _ref_onsets is None
    else _ref_onsets[_ref_onsets < n_times_split] / sfreq
)
print(f"Stimulus onsets in the split window : {len(stimulus_onset_times)}")

---
## Step 7 — Condition-Mean Comparison per Component

Rows = conditions, columns = components, plus a **difference** row (Psilocybin −
Placebo), on the split TF maps. Same scaling rules as the subject-axis variant: one
symmetric limit per column shared by the condition rows (a comparison on two independent
scales is not a comparison), no common limit across columns, the difference row on its
own limit, and every participant put on a common scale first so the means are group
statements rather than a report on the loudest few.

### The topographies get one row, not two

`iva_components` holds **one** topography per `(participant, component)`, shared by both
conditions because there is one mixing matrix per participant. So the topomap figure is
drawn with a **single row** labelled accordingly. Drawing it as a two-row condition
comparison would produce two identical rows and an all-zero difference — a figure that
looks like a null result but is really a statement about the model, not about
psilocybin.

For a genuine topography contrast, use
[`wavelet_iva_channel_joined.ipynb`](wavelet_iva_channel_joined.ipynb), where each
recording has its own mixing matrix.

In [ ]:
# One row: the shared topography. Every participant appears once, under a row label
# that says the map covers both conditions.
SHARED_ROW = " + ".join(CONDITION_ROWS) + " (shared)"

fig_topo = plot_condition_mean_topomaps(
    iva_components,
    participants,
    [SHARED_ROW] * n_subjects,
    [SHARED_ROW],
    iva_info,
    n_channels,
    COMP_INDICES,
    label=LABEL,
    alignment_note=ALIGNMENT_NOTE,
    save_path=(PLOTS_DIR / "shared_mean_topomaps.png") if SAVE_PLOTS else None,
)
plt.show()
plt.close("all")

fig_tf = plot_condition_mean_tf_maps(
    sources_stacked,
    subject_participants,
    subject_conditions,
    CONDITION_ROWS,
    FREQS,
    time_split,
    COMP_INDICES,
    label=LABEL,
    time_marks=stimulus_onset_times if MARK_STIMULUS_ONSETS_ON_TF else None,
    freq_marks=TF_FREQ_MARKS,
    alignment_note=ALIGNMENT_NOTE,
    save_path=(PLOTS_DIR / "condition_mean_tf_maps.png") if SAVE_PLOTS else None,
)
plt.show()
plt.close("all")

if SAVE_PLOTS:
    print(f"Saved condition-mean figures to {PLOTS_DIR}")

---
## Step 8 — Per-Participant Comparison per Component

One figure per component, **Placebo on the first row and Psilocybin on the second**, one
column per participant, columns ordered by **participant ID** — so a participant's two
segments sit directly one above the other and the same participant occupies the same
column in every figure.

The pairing is even tighter here than in the subject-axis variant: the two panels of a
column are two halves of *one* decomposition of *one* recording, with the same mixing
matrix and the same sign. Any difference between them is a difference in the source, with
nothing else varying.

Every panel of a figure shares one colour limit, across participants *and* conditions,
so the whole grid is comparable and the single colourbar means something. A trailing
column holds each condition's mean on its own scale.

The topography counterpart is drawn once, as a single row, for the reason given in Step 7.

In [ ]:
topo_paths = plot_participant_condition_topomaps(
    iva_components,
    participants,
    [SHARED_ROW] * n_subjects,
    [SHARED_ROW],
    iva_info,
    n_channels,
    COMP_INDICES,
    label=LABEL,
    root_dir=PLOTS_DIR / "participants",
    prefix="shared_",
    alignment_note=ALIGNMENT_NOTE,
)
tf_paths = plot_participant_condition_tf_maps(
    sources_stacked,
    subject_participants,
    subject_conditions,
    CONDITION_ROWS,
    FREQS,
    time_split,
    COMP_INDICES,
    label=LABEL,
    root_dir=PLOTS_DIR / "participants",
    time_marks=stimulus_onset_times if MARK_STIMULUS_ONSETS_ON_TF else None,
    freq_marks=TF_FREQ_MARKS,
    alignment_note=ALIGNMENT_NOTE,
)
print(f"Wrote {len(topo_paths)} shared-topography figures and {len(tf_paths)} TF "
      f"figures to {PLOTS_DIR / 'participants'}")

for path in topo_paths + tf_paths:
    display(Image(filename=str(path)))

---
## Step 9 — Stimulus-Locked TF Comparison (Averaged Over Stimuli)

The third view, as in the subject-axis variant: cut a fixed epoch around every stimulus
onset and average it, so the contrast becomes a contrast of **stimulus responses** rather
than of whole tracks. A response locked to onsets is smeared out when the map spans every
stimulus at once.

One thing is genuinely different here, and it matters: **each condition is epoched with
its own onsets on its own segment.** The tracks keep their own time bases in this
variant, so there is no single onset list covering both — `condition_onsets` returns each
condition's, local to its segment, and each split track is averaged against its own.
Both are cut on the same paradigm window from
[`AssrEpoch`](../../src/definitions/constants.py) via `iva_quality.onset_window`, taking
the shorter post-onset span of the two so the two averages share one epoch axis and can
be stacked.

**No baseline subtraction**, as everywhere else: the pre-onset interval reads ≈ 0 by
construction because each track was z-scored over time, so structure there is a warning
sign rather than a response. Keeping the post-offset tail in view is what distinguishes a
response that stops with the stimulus from one that runs on.

Skipped for experiments without stimulus annotations, and when a condition's window is
too short to fit `MIN_ONSETS_FOR_EPOCH_AVERAGE` epochs.

In [ ]:
# Per condition: its own onsets, on its own split segment.
_epoch_info = {}
for name, arr in sources_by_condition.items():
    condition = next(c for c in paired.conditions if c.value == name)
    onsets = onsets_by_condition[condition]
    n_t = arr.shape[-1]
    if onsets is None or len(onsets) == 0:
        _epoch_info[name] = None
        continue
    onsets_in = np.asarray(onsets)[np.asarray(onsets) < n_t].astype(int)
    if onsets_in.size == 0:
        _epoch_info[name] = None
        continue
    pre, post = iva_quality.onset_window(onsets_in, n_t, sfreq)
    n_fitting = int(((onsets_in - pre >= 0) & (onsets_in + post <= n_t)).sum())
    _epoch_info[name] = (onsets_in, pre, post, n_fitting, n_t)
    print(f"{name:<12}: {len(onsets_in)} onset(s) in window, {n_fitting} epoch(s) fit, "
          f"window ({pre} pre, {post} post)")

run_epoch_comparison = all(
    info is not None and info[3] >= MIN_ONSETS_FOR_EPOCH_AVERAGE
    for info in _epoch_info.values()
) and len(_epoch_info) > 0

if not run_epoch_comparison:
    print(
        "\nStimulus-locked comparison skipped: at least one condition has fewer than "
        f"{MIN_ONSETS_FOR_EPOCH_AVERAGE} fitting epoch(s). Raise N_TIMES_SUBSET (or use "
        "an experiment with stimulus annotations)."
    )
else:
    # One epoch geometry for both conditions, so the two averages can be stacked: the
    # shorter post-onset span wins (pre is a paradigm constant).
    EPOCH_PRE = min(info[1] for info in _epoch_info.values())
    EPOCH_POST = min(info[2] for info in _epoch_info.values())
    epoch_times = np.arange(-EPOCH_PRE, EPOCH_POST) / sfreq
    EPOCH_MARKS = [0.0, min(AssrEpoch.STIMULUS_DURATION_S, float(epoch_times[-1]))]

    onset_by_condition = {}
    for name, arr in sources_by_condition.items():
        onsets_in, _pre, _post, _n_fit, _n_t = _epoch_info[name]
        averaged, n_used = iva_quality.epoch_average(
            arr, onsets_in, EPOCH_PRE, EPOCH_POST
        )
        # Reference every FREQUENCY to its own pre-onset mean. The z-scoring puts the
        # pre-onset interval near 0 over the WHOLE recording, but not within any single
        # epoch — the local level still drifts — so this is what turns the map into a
        # change rather than a level. Mean only, no divisor: 1/f is already gone (the
        # data were z-scored per channel-frequency before the decomposition), and at the
        # bottom of the map a 100 ms baseline spans under one cycle, so its SD is mostly
        # wavelet phase and dividing by it would manufacture texture there.
        averaged = iva_quality.subtract_epoch_baseline(averaged, epoch_times < 0.0)
        onset_by_condition[name] = averaged
        print(f"{name:<12}: averaged {n_used} epoch(s) -> {averaged.shape}")

    onset_stacked, onset_participants, onset_conditions = (
        stack_conditions_on_subject_axis(
            onset_by_condition, CONDITION_ROWS, participants
        )
    )
    print(f"\nStacked onset averages : {onset_stacked.shape}  (2P, N_PCA, F, W)")
    print(f"Epoch window           : {EPOCH_PRE + EPOCH_POST} samples "
          f"({EPOCH_PRE} pre, {EPOCH_POST} post) = "
          f"[{epoch_times[0]:.3f}, {epoch_times[-1]:.3f}] s")
    if EPOCH_POST < int(round(EPOCH_POST_S * sfreq)):
        print(f"  NOTE: post-onset span trimmed from {EPOCH_POST_S} s to "
              f"{EPOCH_POST / sfreq:.3f} s by the shortest inter-onset gap.")
    _baseline = np.abs(onset_stacked[:, COMP_INDICES, :, :EPOCH_PRE]).mean()
    _response = np.abs(onset_stacked[:, COMP_INDICES, :, EPOCH_PRE:]).mean()
    print(f"Mean |baseline| / mean |post-onset| : "
          f"{_baseline:.4g} / {_response:.4g}  (ratio {_baseline / _response:.2f})")

### Condition means — onset-averaged

In [ ]:
if run_epoch_comparison:
    fig_tf_onset = plot_condition_mean_tf_maps(
        onset_stacked,
        onset_participants,
        onset_conditions,
        CONDITION_ROWS,
        FREQS,
        epoch_times,
        COMP_INDICES,
        label=LABEL,
        freq_marks=TF_FREQ_MARKS,
        epoch_marks=EPOCH_MARKS,
        alignment_note=ALIGNMENT_NOTE,
        save_path=(
            (PLOTS_DIR / "condition_mean_tf_maps_onset.png") if SAVE_PLOTS else None
        ),
    )
    plt.show()
    plt.close("all")

### Per participant — onset-averaged

Same layout as Step 8, on the stimulus-locked epoch. Written with an `onset_` filename
prefix so these sit alongside the whole-track figures rather than replacing them.

In [ ]:
if run_epoch_comparison:
    onset_tf_paths = plot_participant_condition_tf_maps(
        onset_stacked,
        onset_participants,
        onset_conditions,
        CONDITION_ROWS,
        FREQS,
        epoch_times,
        COMP_INDICES,
        label=LABEL,
        root_dir=PLOTS_DIR / "participants",
        prefix="onset_",
        freq_marks=TF_FREQ_MARKS,
        epoch_marks=EPOCH_MARKS,
        alignment_note=ALIGNMENT_NOTE,
    )
    print(f"Wrote {len(onset_tf_paths)} onset-averaged TF figures to "
          f"{PLOTS_DIR / 'participants'}")
    for path in onset_tf_paths:
        display(Image(filename=str(path)))